In [1]:
!pip install torch transformers accelerate pandas evaluate nltk rouge_score kagglehub

"""
Recipe gpt2 pretrained for fine-tuning.
Model already knows english, so just needs to learn recipes.

Recipes come out in this format:

    <|startofrecipe|><|ingredients|>...<|title|>...<|directions|>...<|endofrecipe|>

Dataset is downloaded from kagglehub and stored on google drive.
Training is checkpointed to google drive to support continuation after colab sessions disconnect.
"""

import os
import re
import ast
import math
import json
import glob
import random

# Only parallelize the workers, otherwise crash
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import pandas as pd
import evaluate
import nltk
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
from transformers.trainer_utils import get_last_checkpoint

# Important Variables
# Made up words added to the vocabulary to mark off the parts of a recipe. The model needs to tell where the ingredients stop and the directions start.
MODEL_NAME = "gpt2" # only used for its tokenizer, not for model weights
BOS_TOKEN = "<|startofrecipe|>"
EOS_TOKEN = "<|endofrecipe|>"
PAD_TOKEN = "<|pad|>"
SPECIAL_TOKENS = {"bos_token": BOS_TOKEN, "eos_token": EOS_TOKEN, "pad_token": PAD_TOKEN}
SECTION_TOKENS = ["<|title|>", "<|ingredients|>", "<|directions|>"]

# Training variables
SAMPLE_SIZE = 250000 # 250k has reasonable runtime, like 6 hrs with 5 epochs, way better results than 50k
KAGGLE_DATASET = "paultimothymooney/recipenlg"
SEED=42

MAX_LENGTH = 512 # good for reasonable recipes
N_EMBD = 512
N_LAYER = 6
N_HEAD = 8 # typical head size seen
N_INNER = 2048 # based on real typical real transformers
# with 5 epochs for training from scratch, builds 45M parameter model getting about 6 hours runtime on colab GPU (switch between 3 google accounts)

# Significantly smaller LR for finetuning (5e-5) or else model destroyed
# Few epochs needed since the model already knows english, also reach approx runtime target
NUM_EPOCHS = 2
LEARNING_RATE =  5e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4 # add up 4 batches before updating, so the real batch is 32
WEIGHT_DECAY=0.01
WARMUP_STEPS = 500 # ease the learning rate up instead of starting flat out
# Checkpoint cadence, ~1000 steps is about 30 min for the pretrained 124M model
SAVE_STEPS = 1000
SAVE_TOTAL_LIMIT = 1 # keep only the newest checkpoint or we run out of space real quick
# Eval cadence, decoupled from saving, ~500 steps is about 15 min
# SAVE_STEPS must be an exact multiple of this (1000/500=2)
EVAL_STEPS = 500
# Cap the eval set so each check during training is quick
EVAL_SUBSET_SIZE = 1000
LOGGING_STEPS = 200

# Generator variables
GEN_TEMPERATURE = 0.7 # 1.0 default gave wonky sentences sometimes, 0.7 seems ok, 0.5 not creative
GEN_TOP_P=0.9 # prune off the wierd word choices

# Google drive mounting
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_BASE = "/content/drive/MyDrive/recipe_gpt2"
RUN_NAME = f"{SAMPLE_SIZE // 1000}k_ingfirst"
OUTPUT_DIR = f"{DRIVE_BASE}/results_pretrained_{RUN_NAME}" # training checkpoints
FINAL_MODEL_DIR = f"{DRIVE_BASE}/model_pretrained_{RUN_NAME}" # final saved model
CACHE_PATH = f"{DRIVE_BASE}/preprocessed_recipes_{RUN_NAME}.json" # serialized recipes (shared)
KAGGLEHUB_CACHE_DIR = f"{DRIVE_BASE}/kagglehub_cache"

# kagglehub download location
os.environ["KAGGLEHUB_CACHE"] = KAGGLEHUB_CACHE_DIR


# Sentence splitters, nltk needs for bleu score
for pkg in ("punkt", "punkt_tab"):
    nltk.download(pkg, quiet=True)

# Mounting google drive into notebook
def ensure_drive_mounted():
    from google.colab import drive

    # Only mount if it isn't already mounted, otherwise Colab complains
    if not os.path.ismount(DRIVE_MOUNT_POINT):
        drive.mount(DRIVE_MOUNT_POINT)

    if not os.path.ismount(DRIVE_MOUNT_POINT):
        raise RuntimeError(
            f"Cant mount drive"
        )

    # Ensure the folder we save everything into exists
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Google Drive mounted, saving to:\n   {DRIVE_BASE}")


# Dataloading setup
def serialize(title, ingredients, directions):
    # Ingredients go first so the model can be prompted with them and asked for the rest (autoregressive)
    return (
        f"{BOS_TOKEN}<|ingredients|>{ingredients}"
        f"<|title|>{title}"
        f"<|directions|>{directions}{EOS_TOKEN}"
    )


def row_to_recipe(row):
    title = str(row["title"]).strip().title()

    # NOTE: ingredients and directions stored as Python lists, soneed literal_eval
    # Ingredients get sorted and lowercased so the prompt format is consistent
    ingredients = ", ".join(sorted(
        str(i).strip().lower() for i in ast.literal_eval(row["ingredients"]) if i))
    directions = " ".join(
        str(d).strip() for d in ast.literal_eval(row["directions"]) if d)

    # Did we actually get all three fields? Skip the row if not
    if not title or not ingredients or not directions:
        return None
    return serialize(title, ingredients, directions)


def download_via_kagglehub():
    # import after mounting google drive
    import kagglehub

    os.makedirs(KAGGLEHUB_CACHE_DIR, exist_ok=True)
    print("Downloading RecipeNLG from kagglehub")
    print(f"Cached on Drive at: {KAGGLEHUB_CACHE_DIR}")
    folder = kagglehub.dataset_download(KAGGLE_DATASET)

    # Full dataset is largest csv
    csvs = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    csvs.sort(key=os.path.getsize, reverse=True)
    print(f"Using {csvs[0]}")
    return csvs[0]


def load_from_csv(path):
    print(f"Reading: {path}")

    # Count total rows
    total_rows = len(pd.read_csv(path, usecols=[0]))
    stride = max(1, total_rows // SAMPLE_SIZE)
    print(f"Dataset has {total_rows:,} rows; sampling every {stride} to get ~{SAMPLE_SIZE:,}")

    recipes = []
    row_index=-1

    # Read 10k rows at a time to not hit memory limits, or we crash again
    for chunk in pd.read_csv(path, chunksize=10000):
        # Map lowercase column names back to whatever case the file actually uses
        cols = {c.lower(): c for c in chunk.columns}
        for _, row in chunk.iterrows():
            row_index += 1

            # Keep only every Nth row.
            if row_index % stride != 0:
                continue
            recipe = row_to_recipe({
                "title": row[cols["title"]],
                "ingredients": row[cols["ingredients"]],
                "directions": row[cols["directions"]],
            })
            if recipe:
                recipes.append(recipe)

            # Got enough? Stop reading, no point parsing the rest of the file
            if len(recipes) >= SAMPLE_SIZE:
                break
        if len(recipes) >= SAMPLE_SIZE:
            break
        print(f"kept {len(recipes):,}/{SAMPLE_SIZE:,}")
    print(f"Loaded {len(recipes):,} recipes")
    return recipes


def load_and_serialize_data():
    # Reuse the cache if a previous run already did everything
    if os.path.exists(CACHE_PATH):
        print(f"Found dataset cache at '{CACHE_PATH}'. Loading...")
        with open(CACHE_PATH) as f:
            return json.load(f)

    # No cache, so download and parse from scratch
    recipes = load_from_csv(download_via_kagglehub())

    # Save it so the next run skips above steps
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    with open(CACHE_PATH, "w") as f:
        json.dump(recipes, f)
    print(f"Cached serialized recipes to {CACHE_PATH}")
    return recipes


class RecipeDataset(Dataset):
    # Recipe dataset with lazy tokenization so that we dont crash the notebook

    def __init__(self, recipe_strings, tokenizer, max_length=MAX_LENGTH):
        self.recipe_strings = recipe_strings
        self.tokenizer = tokenizer
        self.max_length = max_length
        print(f"Prepared {len(recipe_strings):,} recipes for lazy tokenization.")

    def __len__(self):
        return len(self.recipe_strings)

    def __getitem__(self, idx):
        # Tokenize one recipe, pad so alwasy the same shape
        enc = self.tokenizer(
            self.recipe_strings[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0) # (max_length,)
        attention_mask = enc["attention_mask"].squeeze(0)

        # Labels are just the inputs shifted by the model itself, padding gets marked -100 so it can be skipped
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# Model setup
def create_model(vocab_size):
    print("Loading pretrained weights (fine-tuning)...")
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

    # Make room for the recipe tokens added to the tokenizer, the new rows start random and the rest keep their pretrained values
    model.resize_token_embeddings(vocab_size)

    # Real gpt2 architecture (12 layers, d=768) so N_EMBD/N_LAYER/N_HEAD/N_INNER does not actually apply
    total_params = sum(p.numel() for p in model.parameters())
    cfg = model.config
    print(f"Model loaded: {total_params:,} parameters "
          f"({cfg.n_layer} layers, d={cfg.n_embd}, {cfg.n_head} heads, "
          f"ctx={cfg.n_positions}, vocab={vocab_size})")
    return model


def generate_recipe(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # no_grad because inference
    with torch.no_grad():
        output = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=MAX_LENGTH,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids(EOS_TOKEN),
        )

    # Keep the special tokens in the output so the sections can be parsed
    return tokenizer.decode(output[0], skip_special_tokens=False)


class PerplexityLoggingCallback(TrainerCallback):
    # Perplexity is just e to the power of the loss, basically "how many words the model is choosing between"

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        # Do it for the training/eval loss
        for loss_key, ppl_key in (("loss", "perplexity"), ("eval_loss", "eval_perplexity")):
            if loss_key in logs:
                logs[ppl_key] = math.exp(logs[loss_key])


# Units and filler words to ignore when checking if ingredient is used in the directions, try to ignore quatities since not really ingredients
COVERAGE_STOPWORDS = {
    "cup", "cups", "teaspoon", "teaspoons", "tsp", "tablespoon", "tablespoons",
    "tbsp", "ounce", "ounces", "oz", "pound", "pounds", "lb", "lbs", "gram",
    "grams", "g", "kg", "ml", "liter", "large", "small", "medium", "fresh",
    "chopped", "sliced", "diced", "minced", "ground", "package", "can", "cans",
    "and", "or", "of", "to", "the", "a", "for",
}


def ingredient_mentioned(ingredient, directions_lower):
    # This function checks whether an ingredient actually got used, for example just "cream" is sufficient to match "1 cup heavy cream"

    # Keep only actual words, skip abbreviations and stuff
    words = [w for w in re.findall(r"[a-z]+", ingredient)
             if len(w) >= 4 and w not in COVERAGE_STOPWORDS]

    # Nothing left after filtering? Try matching on any word instead
    if not words:
        words = re.findall(r"[a-z]+", ingredient)
    return any(w in directions_lower for w in words)


def evaluate_generations(model, tokenizer, eval_strings, sample_size=50):
    # Prompt with everything up to and including <|title|>, so the model has to produce the title and directions
    # Scores the generated directions against the real ones with rouge and bleu
    # Also calculate ingredient coverage which is the fraction of the given ingredients that actually show up

    print(f"\nEvaluating generation quality on {sample_size} unseen samples...")
    print("Task: ingredients -> title + directions (ingredients-only prompt)")
    model.eval()
    rouge_metric = evaluate.load("rouge")

    generated_texts, reference_texts, coverage_scores = [], [], []

    for recipe in eval_strings[:sample_size]:
        # Cut the prompt off right after <|title|>, since the goal is whats generated after
        title_pos = recipe.index("<|title|>") + len("<|title|>")
        dir_pos = recipe.index("<|directions|>") + len("<|directions|>")
        prompt = recipe[:title_pos]
        reference_dir = recipe[dir_pos:].replace(EOS_TOKEN, "").strip()

        # Pull the ingredients back out of the prompt for the coverage metric
        ing_section = re.search(r"<\|ingredients\|>(.*?)(?=<\|title\|>)", recipe).group(1)
        input_ingredients = [i.strip().lower() for i in ing_section.split(",") if i.strip()]

        generated_full = generate_recipe(model, tokenizer, prompt)

        # Parse the directions out of what the model wrote. Generation is sampled, no guarantee the tag is even there
        match = re.search(r"<\|directions\|>(.*?)(<\|endofrecipe\|>|$)", generated_full)
        gen_dir = match.group(1).strip() if match else ""

        # How many of the ingredients did it actually use?
        gen_lower = gen_dir.lower()
        matched = [ing for ing in input_ingredients if ingredient_mentioned(ing, gen_lower)]
        coverage = len(matched) / len(input_ingredients) if input_ingredients else 0.0

        generated_texts.append(gen_dir)
        reference_texts.append(reference_dir)
        coverage_scores.append(coverage)

    # rouge measures word overlap against the reference directions
    rouge_results = rouge_metric.compute(predictions=generated_texts, references=reference_texts)

    # bleu 4 does the same but on 4 word sequence (kind of useless tbh)
    smoothing = nltk.translate.bleu_score.SmoothingFunction().method4
    bleu_scores = [
        nltk.translate.bleu_score.sentence_bleu(
            [nltk.word_tokenize(ref)], nltk.word_tokenize(gen), smoothing_function=smoothing
        )
        for gen, ref in zip(generated_texts, reference_texts)
    ]
    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    avg_coverage = sum(coverage_scores) / len(coverage_scores)

    print("\nFinal Generative Evaluation Results (ingredients -> title + directions):")
    print(f" ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f" ROUGE-L: {rouge_results['rougeL']:.4f}")
    print(f" BLEU-4:  {avg_bleu:.4f}")
    print(f" Ingredient Coverage Rate: {avg_coverage * 100:.2f}%")
    return {
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu4": avg_bleu,
        "ingredient_coverage": avg_coverage,
        "num_eval_samples": len(generated_texts),
    }


def save_training_report(trainer, out_dir, final_metrics, extra_config):
    #Produces three files:
    # training_history.csv - every logged step: train loss, eval loss, perplexity
    # training_summary.json - config plus final/best metrics for scripts
    # training_summary.txt - same but easier to read

    os.makedirs(out_dir, exist_ok=True)
    history = trainer.state.log_history

    # Collapse the log history into one row per step
    rows = {}
    for entry in history:
        step = entry.get("step")
        if step is None:
            continue
        row = rows.setdefault(step, {"step": step})
        for k in ("loss", "eval_loss", "perplexity", "eval_perplexity",
                  "learning_rate", "epoch"):
            if k in entry:
                row[k] = entry[k]

    # Write it out as csv sorted by step
    cols = ["step", "epoch", "loss", "perplexity", "eval_loss",
            "eval_perplexity", "learning_rate"]
    csv_path = os.path.join(out_dir, "training_history.csv")
    with open(csv_path, "w") as f:
        f.write(",".join(cols) + "\n")
        for step in sorted(rows):
            f.write(",".join(str(rows[step].get(c, "")) for c in cols) + "\n")

    # Find the lowest eval loss over the whole run
    evals = [(e["step"], e["eval_loss"]) for e in history if "eval_loss" in e]
    best_step, best_eval_loss = min(evals, key=lambda x: x[1])
    best_ppl=math.exp(best_eval_loss)
    final_eval_loss = evals[-1][1]
    final_ppl = math.exp(final_eval_loss)

    # For later parsing in more scripts if needed
    summary = {
        "config": extra_config,
        "total_steps": trainer.state.global_step,
        "final_eval_loss": final_eval_loss,
        "final_eval_perplexity": final_ppl,
        "best_step": best_step,
        "best_eval_loss": best_eval_loss,
        "best_eval_perplexity": best_ppl,
        "generation_metrics": final_metrics,
    }
    with open(os.path.join(out_dir, "training_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # For easy reading
    with open(os.path.join(out_dir, "training_summary.txt"), "w") as f:
        f.write("TRAINING SUMMARY\n" + "=" * 40 + "\n")
        f.write("Config:\n")
        for k, v in extra_config.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"\nTotal steps: {trainer.state.global_step}\n")
        f.write(f"Final eval loss: {final_eval_loss:.4f} (perplexity {final_ppl:.3f})\n")
        f.write(f"Best  eval loss: {best_eval_loss:.4f} (perplexity {best_ppl:.3f}) "
                f"at step {best_step}\n")
        f.write("\nGeneration metrics:\n")
        for k, v in final_metrics.items():
            f.write(f"  {k}: {v}\n")

    print(f"Saved training report (history CSV + summary JSON/TXT) to {out_dir}")


def main():
    # Guarantee drive is really mounted before anything writes to it
    ensure_drive_mounted()

    # Build the tokenizer
    print("Initializing tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.add_special_tokens(SPECIAL_TOKENS)
    tokenizer.add_tokens(SECTION_TOKENS)
    print(f"Tokenizer vocabulary size: {len(tokenizer)}")

    # Build the model
    model = create_model(vocab_size=len(tokenizer))

    # Tell the model which token ids mean start, end, padding
    model.config.bos_token_id = tokenizer.convert_tokens_to_ids(BOS_TOKEN)
    model.config.eos_token_id = tokenizer.convert_tokens_to_ids(EOS_TOKEN)
    model.config.pad_token_id = tokenizer.pad_token_id

    # Load data
    recipes = load_and_serialize_data()

    random.Random(SEED).shuffle(recipes)
    split_idx=int(len(recipes) * 0.95)
    train_strings, eval_strings = recipes[:split_idx], recipes[split_idx:]
    print(f"\nDataset split: {len(train_strings):,} train / {len(eval_strings):,} eval")

    # Cap the eval set used during training for speed, the full evaluation is used for the final generation metrics
    eval_subset = eval_strings[:EVAL_SUBSET_SIZE]
    print(f"Using {len(eval_subset):,} recipes for periodic eval during training.")

    train_dataset = RecipeDataset(train_strings, tokenizer)
    eval_dataset = RecipeDataset(eval_subset, tokenizer)

    # Trainer
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS, # loss readings, no drive write
        save_strategy="steps",
        save_steps=SAVE_STEPS, # checkpoints which write to drive
        save_total_limit=SAVE_TOTAL_LIMIT, # dont do too much, we run out of space
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        num_train_epochs=NUM_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        fp16=torch.cuda.is_available(), # half precision for 2x speed for GPU
        dataloader_num_workers=2, # parallel workers to hide tokenization cost
        dataloader_persistent_workers=True, # don't re-fork the workers every epoch
        report_to="none",
        load_best_model_at_end=True, # finish on the best checkpoint, not the last
        metric_for_best_model="eval_loss",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        callbacks=[PerplexityLoggingCallback()],
    )

    # Automatically resume from a checkpoint if found
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
    if last_checkpoint:
        print(f"\nResuming from checkpoint: {last_checkpoint}")
    else:
        print("\nNo checkpoint found - starting a fresh run...")
    print(f"Context window: {MAX_LENGTH} tokens | {SAMPLE_SIZE:,} recipes x {NUM_EPOCHS} epochs")

    trainer.train(resume_from_checkpoint=last_checkpoint)

    # Save the finished model and its tokenizer together for fast reloading
    print(f"\nSaving final model to {FINAL_MODEL_DIR}...")
    os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
    model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

    # Score the real generation task now that training is done
    final_metrics = evaluate_generations(model, tokenizer, eval_strings, sample_size=50)

    # Dump the loss/perplexity history and the metrics summary into the model folder so all the reporting data ends up in one place
    save_training_report(
        trainer, FINAL_MODEL_DIR, final_metrics,
        extra_config={
            "run_name": RUN_NAME,
            "base_model": MODEL_NAME,
            "sample_size": SAMPLE_SIZE,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "max_length": MAX_LENGTH,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
        },
    )

    # Hand it a few ingredients and print whatever it comes up with
    print("\nInference demo...")
    demo_prompt = (
        f"{BOS_TOKEN}<|ingredients|>avocado, cocoa powder, coffee, heavy cream, sugar"
        "<|title|>"
    )
    print("\nGenerated Recipe Result:")
    print(generate_recipe(model, tokenizer, demo_prompt))


if __name__ == "__main__":
    main()

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=d9c75b9608f25a17a3a470e1f3aad216047ba394923b9ce6d1b9313e806a096d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
Mounted at /content/drive
Google Drive mounted, saving to:
   /content/drive/MyDrive/recipe_gpt2
Initializing tokenizer...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss



Saving final model to /content/drive/MyDrive/recipe_gpt2/model_pretrained_250k_ingfirst...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating generation quality on 50 unseen samples...
Task: ingredients -> title + directions (ingredients-only prompt)



Final Generative Evaluation Results (ingredients -> title + directions):
 ROUGE-1: 0.3209
 ROUGE-L: 0.2165
 BLEU-4:  0.0493
 Ingredient Coverage Rate: 59.31%
Saved training report (history CSV + summary JSON/TXT) to /content/drive/MyDrive/recipe_gpt2/model_pretrained_250k_ingfirst

Inference demo...

Generated Recipe Result:
<|startofrecipe|><|ingredients|>avocado, cocoa powder, coffee, heavy cream, sugar<|title|>Coffee And Avocado<|directions|>Cream the avocado. Add the coffee, heavy cream, sugar and cocoa powder. Mix well. Pour into glasses and enjoy.<|endofrecipe|>
